# Evaluating the Equity of Urban Heat Mitigation in Transit-Oriented Development Districts in Maryland
### An assessment of tree canopy, land surface temperature, and environmental justice

**Author:** Siddhi Pawar <br>
**Course:** URSP688Y: Urban Data Science & Smart Cities <br>
**Professor:** Dr. Chester Harvey <br>
**Date:** April 29, 2026 <br>

--- 

## Research Question
How are environmental conditions, including land surface temperature (LST) and tree canopy coverage, associated with the level of transit-oriented development (TOD) exposure within census tracts across Maryland, and how do these patterns differ across socioeconomic characteristics?

TOD exposure is measured as the proportion of each census tract's area that overlaps with Maryland's state-defined TOD boundaries. Proportions will be classified as tracts with no TOD exposure (area % = 0), tracts with low exposure (0 < area % < 50), and tracts with high exposure (area % >= 50). 

--- 

## Approach / Pseudocode

### ArcGIS Pro Analysis:  <br>
**PHASE 1 - ArcGIS Spatial Data Setup** <br>
    **1a.** Add Maryland state boundary, Maryland designated TOD, and Census tract (ACS 2024) shapefiles <br>
    **1b.** Add LST Landsat 8/9 scene, NLCD CONUS tree canopy coverage, and NLCD fractional impervious surface rasters <br>
    **1c.** Project everything to NAD 1983 StatePlan Maryland FIPS 1900 (US Feet) <br>
    **1d.** Convert the LST raster from Kelvin to Fahrenheit (raster calculator) <br>
    **1e.** Extract the tree canopy and impervious surface raster to the mask extent of the Maryland boundary (to concentrate the analysis scale to the state) <br>
    **1f.** Use the Zonal Statistics tool to gather LST, tree canopy, and impervious surface data for the Census tracts <br>
    **1g.** Export CSVs and shapefiles <br>

### Python Analysis:  <br>

**PHASE 2 - Python Spatial Data Setup** <br>
    **2a.** Load Maryland TOD boundaries and Census tracts <br>
    **2b.** Confirm projections and project again if necessary <br>
    **2c.** Create the TOD exposure distinctions/classifications  --> <br>
            - Intersect the Census tracts with the TOD polygons <br>
            - Divide the area of the intersection with the total area of the Census tract <br>
            - Assign a value of 0 to tracts with no overlap <br>
    **2d.** Classify the tracts for mapping and summary purposes <br>
            - No TOD: 0% exposure or overlap of tract <br>
            - Low TOD: 0% < exposure or overlap of tract <= 50% <br>
            - High TOD: exposure or overlap of tract > 50% <br>

**PHASE 3 - Demographic Data** <br>
    **3a.** Pull 2024 5-year estimates ACS data through a Census API <br>
            - Median household income <br>
            - Total population <br>
            - Race and ethnicity <br>
    **3b.** Merge the ACS data to the tract GEOIDs <br>
    **3c.** Calculate population density for each Census tract <br>

**PHASE 4 - Environmental Data Integration** <br>
    **4a.** Load the zonal statistics CSVs from ArcGIS Pro to this notebook <br>
            - LST CSV = mean LST (in fahrenheit) for each Census tract <br>
            - Tree canopy CSV = mean tree canopy (%) for each Census tract <br>
            - Impervious surface CSV = mean impervious surface (%) for each Census tract <br>
    **4b.** Make sure that the GEOID for the tracts match up <br>
    **4c** Merge all three CSVs to a tract dataframe <br>

**PHASE 5 - Analysis and Visualization** <br>
    **5a.** Create some sort of summary statistics table that showcases LST, tree canopy, and impervious surfae data by TOD exposure type <br>
    **5b.** Create some sort of summary statistics table that showcases median household income and racial data by TOD exposure type <br>
    **5c.** Create scatterplots for... <br>
            - Tree canopy vs LST for TOD exposure <br>
            - TOD exposure area type vs LST <br>
            - Income vs Tree canopy <br>
            - Race vs Tree canopy <br>
    **5d.** Create maps to spatially show these relationships <br>
            - TOD exposure types across Maryland <br>
            - LST, tree canopy, income, and race across Maryland <br>
    













In [23]:
#Import the necessary packages (this will be updated as needed)
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [24]:
#Start the spatial setup of the Census tracts and TOD files

#Load Maryland Census tracts (ACS 2024)
census_tracts = gpd.read_file("2024_Census_tracts/2024_Census_Tracts.shp")

#Load the Maryland TOD boundaries
TOD = gpd.read_file("MD_Designated_TODs/MD_Designated_TODs.shp")

#Confirm that both files have loaded properly
print("Census tracts:", len(census_tracts))
print("Maryland TOD districts:", len(TOD))

#Display the Census tract information
census_tracts.head()

Census tracts: 1475
Maryland TOD districts: 17


,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,Shape_Leng,Shape_Area,geometry
0,24,005,403100,24005403100,1400000US24005403100,4031,Census Tract 4031,G5020,S,1913598.0,0.0,+39.3530950,-076.7333148,24386.404860,2.059710e+07,"POLYGON ((1384461.837 616752.682, 1384630.936 ..."
1,24,005,403201,24005403201,1400000US24005403201,4032.01,Census Tract 4032.01,G5020,S,1764534.0,0.0,+39.3549038,-076.7215235,21171.869169,1.899260e+07,"POLYGON ((1387632.604 617558.335, 1387855.691 ..."
2,24,033,807304,24033807304,1400000US24033807304,8073.04,Census Tract 8073.04,G5020,S,1776714.0,0.0,+39.0249780,-076.9594342,19996.981133,1.912254e+07,"POLYGON ((1320372.199 492350.448, 1320392.078 ..."
3,24,033,807305,24033807305,1400000US24033807305,8073.05,Census Tract 8073.05,G5020,S,3030479.0,4428.0,+39.0121779,-076.9635510,24370.659553,3.266427e+07,"POLYGON ((1318789.779 489571.471, 1318828.406 ..."
4,24,033,807405,24033807405,1400000US24033807405,8074.05,Census Tract 8074.05,G5020,S,7093435.0,27200.0,+39.0383526,-076.9283090,36212.038201,7.663847e+07,"POLYGON ((1326579.214 493631.119, 1326698.43 4..."


In [25]:
#Display the TOD information
TOD.head()

,OBJECTID_1,Station_Na,Region,County,Des_Catego,Shape__Are,Shape__Len,GlobalID,Shape_Leng,Shape_Area,geometry
0,2,State Center Metro,Baltimore,Baltimore City,Maryland Designated TOD Station,1.729127e+05,1954.844917,c8f987d1-5f89-4a31-bb42-368687374977,4958.006146,1.112942e+06,"POLYGON ((1419465.644 596569.96, 1419590.554 5..."
1,9,New Carrollton Metro MARC/Amtrak Station,Washington,Prince George's County,Maryland Designated TOD Station,8.766574e+05,6842.647577,8ce16cad-af88-49fc-8a6c-ad983e669d0b,17453.100508,5.698684e+06,"MULTIPOLYGON (((1348441.54 465381.441, 1347616..."
2,10,Naylor Road Metro Station,Washington,Prince George's County,Maryland Designated TOD Station,4.381490e+05,2872.619482,1e5abf3e-b7dd-4082-beb1-52f858e568a7,7338.417298,2.856099e+06,"POLYGON ((1325429.206 429568.781, 1324858.394 ..."
3,11,Branch Avenue Metro Station,Washington,Prince George's County,Maryland Designated TOD Station,1.057239e+06,6465.684320,a9040d47-f2cf-4620-b1df-b9490bf05f24,16513.888581,6.895492e+06,"MULTIPOLYGON (((1336224.49 423521.653, 1336555..."
4,13,Aberdeen MARC Station,Baltimore,Harford County,Maryland Designated TOD Station,2.074145e+06,7094.502524,82b1bd60-cf6e-4e1c-8054-23d8fe6f2280,17947.616029,1.327254e+07,"POLYGON ((1549099.533 671266.063, 1548191.092 ..."


In [26]:
#Confirm the projections/coordinate systems of the Census tract and TOD files
#Both layers were projected in ArcGIS Pro

print("Census Tract CRS:", census_tracts.crs)
print("Maryland TOD CRS:", TOD.crs)

#If these do not align, they should be reporjected to match
if census_tracts.crs != TOD.crs:
    census_tracts = census_tracts.to_crs(TOD.crs)
    print("The Census tracts have been reprojected to match the TODs")
else:
    print("\nThe CRS match!")

Census Tract CRS: PROJCS["NAD83 / Maryland (ftUS)",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",37.6666666666667],PARAMETER["central_meridian",-77],PARAMETER["standard_parallel_1",38.3],PARAMETER["standard_parallel_2",39.45],PARAMETER["false_easting",1312333.33333333],PARAMETER["false_northing",0],UNIT["US survey foot",0.304800609601219,AUTHORITY["EPSG","9003"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Maryland TOD CRS: PROJCS["NAD83 / Maryland (ftUS)",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",37.6666666666667],PARAMETER["central_m

In [27]:
#Create the TOD exposure classifications through proportions
#Instead of a binary TOD vs non-TOD comparison, the use of exposure areas will look at what proportion of each Census tract overlaps with a TOD district
#and then will be classified based on the overlap area

census_tracts["tract_area"] = census_tracts.geometry.area

#Intersect the tracts with the TOD polygons
intersection = gpd.overlay(census_tracts, TOD, how = "intersection")

#Calculate the area of the intersected pieces/portions
intersection["intersection_area"] = intersection.geometry.area

#Add or sum up the overlay area for each Census tract with the GEOID
TOD_area = (intersection.groupby("GEOID")["intersection_area"].sum().reset_index())

#Now that there is an overlap established, merge the overlap areas to the Census tracts
census_tracts = census_tracts.merge(TOD_area, on = "GEOID", how = "left")

#If there are tracts with no TOD overlap, they can be given a 0 value
census_tracts["intersection_area"] = census_tracts["intersection_area"].fillna(0)

#Calculate the TOD proportions/exposure for the Census tracts
census_tracts["percent_TOD"] = (census_tracts["intersection_area"] / census_tracts["tract_area"]) * 100

#Display the results
print("Tracts with no TOD exposure:", (census_tracts["percent_TOD"] == 0).sum())
print("Tracts with TOD exposure:", (census_tracts["percent_TOD"] > 0).sum())
print("Tracts with Low TOD exposure (0–50%):", ((census_tracts["percent_TOD"] > 0) & (census_tracts["percent_TOD"] <= 50)).sum())
print("Tracts with High TOD exposure (> 50%):", (census_tracts["percent_TOD"] > 50).sum())

census_tracts[["GEOID", "percent_TOD"]].head(10)


Tracts with no TOD exposure: 1423
Tracts with TOD exposure: 52
Tracts with Low TOD exposure (0–50%): 50
Tracts with High TOD exposure (> 50%): 2


,GEOID,percent_TOD
0,24005403100,0.0
1,24005403201,0.0
2,24033807304,0.0
3,24033807305,0.0
4,24033807405,0.0
5,24033807407,0.0
6,24033803700,0.0
7,24027606603,0.0
8,24005400701,0.0
9,24005400702,0.0
